# Credit-Adjusted Valuation and Market Comparison
**UniCredit Variable Rate Bond 2034 (ISIN IT0005599110) | Trade date: 5 Nov 2025**

Builds the risky (CVA-adjusted) valuation from the flat-hazard credit model, then compares the risk-free and risky fair values against the observed EuroTLX market price and backs out an implied CDS spread.

## CVA-adjusted valuation

Builds the full coupon exposure schedule and accumulates the credit valuation adjustment against the risk-free price computed above.

In [1]:
import os, sys, math
import numpy as np
import pandas as pd

_here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
sys.path.insert(0, _here)
sys.path.insert(0, os.path.join(_here, os.pardir, "helpers"))  # shared fi_* modules

from fi_calendar import (
    load_target_holidays, modified_following, add_business_days,
    days_30_360, yearfrac_act_360, yearfrac_act_365,
)
from fi_curve import load_curve, make_df_fn
from fi_bond import (
    build_schedule, load_vol_surface, get_flat_vol, price_bond, accrued_interest,
    TRADE_DATE, SPOT_LAG, ISSUE_DATE, MATURITY_DATE,
    NOTIONAL, PARTICIPATION, CURRENT_COUPON_RATE, CURRENT_PERIOD_START,
    K_EFF_CAP, K_EFF_FLOOR,
)
from fi_credit import price_risky_bond, survival_prob

HOLIDAYS_FILE       = os.path.join(_here, os.pardir, "data", "Holidays.xlsx")
TERM_STRUCTURE_FILE = os.path.join(_here, os.pardir, "data", "Interp_term_structure.xlsx")
VOL_SURFACE_FILE    = os.path.join(_here, os.pardir, "data", "Shifted_Black_vol_surface.xlsx")

# ── Load market data and build the SAME pricing inputs as Q5 ────────────────
# (Q6/Q10 used to redo this from scratch with raw QuantLib and a hand-typed
# date schedule, in three separate places; now there is one shared setup.)
holidays    = load_target_holidays(HOLIDAYS_FILE)
curve       = load_curve(TERM_STRUCTURE_FILE)
vol_surface = load_vol_surface(VOL_SURFACE_FILE, "6M")
spot_date   = add_business_days(TRADE_DATE, SPOT_LAG, holidays)
schedule    = build_schedule(ISSUE_DATE, MATURITY_DATE, holidays)
df_fn       = make_df_fn(curve, spot_date)

mat_adj   = modified_following(MATURITY_DATE, holidays)
cap_mat_y = yearfrac_act_365(spot_date, mat_adj)
sigma_cap = get_flat_vol(vol_surface, cap_mat_y, K_EFF_CAP)
sigma_flr = get_flat_vol(vol_surface, cap_mat_y, K_EFF_FLOOR)

AI = accrued_interest(TRADE_DATE, CURRENT_PERIOD_START, CURRENT_COUPON_RATE, NOTIONAL)

print("Modules loaded: fi_calendar | fi_curve | fi_bond | fi_credit")
print(f"Trade date : {TRADE_DATE}   Spot date : {spot_date}")
print(f"Cap/floor maturity : {cap_mat_y:.4f}Y")


Modules loaded: fi_calendar | fi_curve | fi_bond | fi_credit
Trade date : 2025-11-05   Spot date : 2025-11-07
Cap/floor maturity : 8.6000Y


In [2]:
# =============================================================================
# Q6 -- Credit Valuation Adjustment (CVA)
# =============================================================================
# UniCredit 5Y EUR CDS, trade date 05-Nov-2025 (UNIC5YEUAM=R)
CDS_SPOT_BP   = 38.78
RECOVERY_BASE = 0.40

# Risk-free price -- the SAME Displaced Black engine as Q5, called directly
# rather than hand-copied as a literal, so this notebook cannot silently
# drift from Q5 if the curve, vol surface or bond terms ever change.
rf_result = price_bond(
    df_fn=df_fn, holidays=holidays, schedule=schedule, spot_date=spot_date,
    sigma_cap=sigma_cap, sigma_flr=sigma_flr,
)
rf_npv = rf_result["gross_price"]

# Risky price and CVA via fi_credit (dominant notional-loss term; the
# smaller coupon-loss term is a disclosed simplification, not coded).
risky_result = price_risky_bond(
    df_fn=df_fn, schedule=schedule, spot_date=spot_date, holidays=holidays,
    cds_bp=CDS_SPOT_BP, sigma_cap=sigma_cap, sigma_flr=sigma_flr,
    recovery=RECOVERY_BASE,
)
cva_total   = risky_result["cva"]
risky_npv   = risky_result["risky_gross"]
risky_clean = risky_npv - AI
lam_base    = (CDS_SPOT_BP / 1e4) / (1.0 - RECOVERY_BASE)
tau_40      = yearfrac_act_365(spot_date, mat_adj)

# -----------------------------------------------------------------------
# Coupon-vs-principal split of the SAME dominant-term CVA sum, for
# reporting only -- reproduces fi_credit's per-period terms exactly, and
# is asserted below to reconcile to fi_credit's own total (single source
# of truth: this is a breakdown of price_risky_bond's number, not a
# second, independent CVA calculation).
# -----------------------------------------------------------------------
cashflow_dates = [(p["Payment Date"], False) for p in schedule if p["Payment Date"] > spot_date]
cashflow_dates.append((mat_adj, True))
cashflow_dates.sort(key=lambda x: x[0])

cva_coupons = cva_principal = 0.0
Q_prev = 1.0
cva_rows = []
for pay_date, is_principal in cashflow_dates:
    T_i   = yearfrac_act_365(spot_date, pay_date)
    df_i  = df_fn(pay_date)
    Q_i   = survival_prob(CDS_SPOT_BP, T_i, RECOVERY_BASE)
    dQ    = Q_prev - Q_i
    cva_i = (1.0 - RECOVERY_BASE) * NOTIONAL * dQ * df_i
    if is_principal:
        cva_principal += cva_i
    else:
        cva_coupons += cva_i
    cva_rows.append({
        "Payment Date": pay_date.isoformat(), "tau_i": round(T_i, 4),
        "Z_rf": round(df_i, 6), "Q(T_{i-1})": round(Q_prev, 6),
        "Q(T_i)": round(Q_i, 6), "dQ": round(dQ, 6), "CVA_i EUR": round(cva_i, 4),
    })
    Q_prev = Q_i

cva_detail = pd.DataFrame(cva_rows)
_recon_diff = abs((cva_coupons + cva_principal) - cva_total)
assert _recon_diff < 1e-6, f"CVA coupon/principal split does not reconcile to fi_credit total (diff={_recon_diff:.2e})"

print("=" * 66)
print("Q6 -- Credit Valuation Adjustment (CVA)")
print(f"   Trade date : {TRADE_DATE}   |   Spot date : {spot_date}")
print("=" * 66)

print(f"\nModel parameters")
print(f"  CDS spread (flat)  : {CDS_SPOT_BP:.2f} bp  (UNIC5YEUAM=R, 05-Nov-2025)")
print(f"  Recovery rate      : {RECOVERY_BASE*100:.0f}%")
print(f"  Hazard rate lambda : {lam_base:.6f} p.a.")
print(f"  Survival prob T40  : {math.exp(-lam_base*tau_40):.4f}  "
      f"(cum. default prob: {(1-math.exp(-lam_base*tau_40))*100:.2f}%)")

print(f"\nCVA breakdown")
print(f"  CVA (coupons)   : EUR {cva_coupons:.4f}  ({cva_coupons/cva_total*100:.1f}% of total CVA)")
print(f"  CVA (principal) : EUR {cva_principal:.4f}  ({cva_principal/cva_total*100:.1f}% of total CVA)")
print(f"  CVA (total)     : EUR {cva_total:.4f}  ({cva_total/rf_npv*100:.2f}% of risk-free NPV)")
print(f"  Reconciliation  : coupon+principal split vs fi_credit total -- diff = {_recon_diff:.2e}")

print(f"\nValuation summary  (EUR per 1000 nominal)")
print(f"  {'Risk-free NPV (gross)':<30} EUR {rf_npv:>8.4f}  ({rf_npv/10:.4f}%)")
print(f"  {'CVA':<30} EUR {cva_total:>8.4f}  ({cva_total/rf_npv*100:.4f}%)")
print(f"  {'Risky NPV (gross)':<30} EUR {risky_npv:>8.4f}  ({risky_npv/10:.4f}%)")
print(f"  {'Accrued interest':<30} EUR {AI:>8.4f}")
print(f"  {'Risky clean price':<30} EUR {risky_clean:>8.4f}  ({risky_clean/10:.4f}%)")

print("\nPer-period CVA detail:")
print(cva_detail.to_string(index=False))


Q6 -- Credit Valuation Adjustment (CVA)
   Trade date : 2025-11-05   |   Spot date : 2025-11-07

Model parameters
  CDS spread (flat)  : 38.78 bp  (UNIC5YEUAM=R, 05-Nov-2025)
  Recovery rate      : 40%
  Hazard rate lambda : 0.006463 p.a.
  Survival prob T40  : 0.9459  (cum. default prob: 5.41%)

CVA breakdown
  CVA (coupons)   : EUR 29.2888  (100.0% of total CVA)
  CVA (principal) : EUR 0.0000  (0.0% of total CVA)
  CVA (total)     : EUR 29.2888  (2.66% of risk-free NPV)
  Reconciliation  : coupon+principal split vs fi_credit total -- diff = 3.55e-15

Valuation summary  (EUR per 1000 nominal)
  Risk-free NPV (gross)          EUR 1102.2711  (110.2271%)
  CVA                            EUR  29.2888  (2.6571%)
  Risky NPV (gross)              EUR 1072.9823  (107.2982%)
  Accrued interest               EUR   4.7794
  Risky clean price              EUR 1068.2029  (106.8203%)

Per-period CVA detail:
Payment Date  tau_i     Z_rf  Q(T_{i-1})   Q(T_i)       dQ  CVA_i EUR
  2025-12-12 0.0959 0.

In [3]:
# =============================================================================
# Save CVA detail and print final summary
# =============================================================================
cva_detail.to_csv(os.path.join(_here, os.pardir, "data", "q6_cva_detail.csv"), index=False)
print("Saved: q6_cva_detail.csv")

print("\n" + "=" * 66)
print(f"FINAL SUMMARY  (EUR per 1000 nominal, CDS={CDS_SPOT_BP:.2f}bp, R=40%)")
print("=" * 66)
print(f"{'Item':<35} {'EUR':>10} {'%':>8}")
print("-" * 56)
print(f"{'Risk-free NPV (gross)':<35} {rf_npv:>10.4f} {rf_npv/10:>8.4f}")
print(f"{'CVA - coupons':<35} {cva_coupons:>10.4f} {cva_coupons/rf_npv*100:>8.4f}")
print(f"{'CVA - principal':<35} {cva_principal:>10.4f} {cva_principal/rf_npv*100:>8.4f}")
print(f"{'CVA - total':<35} {cva_total:>10.4f} {cva_total/rf_npv*100:>8.4f}")
print(f"{'Risky NPV (gross)':<35} {risky_npv:>10.4f} {risky_npv/10:>8.4f}")
print(f"{'Accrued interest':<35} {AI:>10.4f}")
print(f"{'Risky clean price':<35} {risky_clean:>10.4f} {risky_clean/10:>8.4f}")


Saved: q6_cva_detail.csv

FINAL SUMMARY  (EUR per 1000 nominal, CDS=38.78bp, R=40%)
Item                                       EUR        %
--------------------------------------------------------
Risk-free NPV (gross)                1102.2711 110.2271
CVA - coupons                          29.2888   2.6571
CVA - principal                         0.0000   0.0000
CVA - total                            29.2888   2.6571
Risky NPV (gross)                    1072.9823 107.2982
Accrued interest                        4.7794
Risky clean price                    1068.2029 106.8203


## Market price comparison and implied CDS spread

Reuses `rf_npv`, `schedule`, `df_fn`, `holidays`, `spot_date`, `sigma_cap`,
`sigma_flr` and `AI` computed above -- the same Displaced Black + CVA
engine as Q5/Q6, not a second, independent re-derivation -- then compares
the resulting fair values to the observed EuroTLX market price and solves
for the CDS spread the market is implying.

In [4]:
# =============================================================================
# Q10 -- Market Price Comparison and Implied CDS Spread
# =============================================================================
from scipy.optimize import brentq

def compute_risky_gross(cds_spread_dec: float, recovery: float = 0.40) -> float:
    """Risky gross price at an arbitrary CDS spread (decimal, e.g. 0.0072 = 72bp),
    via the same fi_credit engine used in Q6 -- not a re-implementation."""
    return price_risky_bond(
        df_fn=df_fn, schedule=schedule, spot_date=spot_date, holidays=holidays,
        cds_bp=cds_spread_dec * 1e4, sigma_cap=sigma_cap, sigma_flr=sigma_flr,
        recovery=recovery,
    )["risky_gross"]

rf_npv_clean = rf_npv - AI
print(f"\nRisk-free NPV  gross : EUR {rf_npv:.4f}  ({rf_npv / 10:.4f}%)")
print(f"Risk-free NPV  clean : EUR {rf_npv_clean:.4f}  ({rf_npv_clean / 10:.4f}%)")
print(f"Accrued interest     : EUR {AI:.4f}")

# =============================================================================
# Market price inputs
# =============================================================================
MKT_CLEAN_PCT = 100.81          # % of par  (EuroTLX, 05-Nov-2025)
MKT_CLEAN_EUR = NOTIONAL * MKT_CLEAN_PCT / 100.0
MKT_GROSS_EUR = MKT_CLEAN_EUR + AI

CDS_MARKET_BP = 72.0             # 5Y UniCredit CDS (investing.com) -- a
                                  # different quote source than Q6's
                                  # UNIC5YEUAM=R, used deliberately here
                                  # as an independent cross-check input.
RECOVERY_MKT  = 0.40

risky_gross_q6 = compute_risky_gross(CDS_MARKET_BP / 1e4, RECOVERY_MKT)
risky_clean_q6 = risky_gross_q6 - AI

print(f"\nRisky NPV at CDS={CDS_MARKET_BP:.0f}bp   gross : EUR {risky_gross_q6:.4f}  ({risky_gross_q6 / 10:.4f}%)")
print(f"Risky NPV at CDS={CDS_MARKET_BP:.0f}bp   clean : EUR {risky_clean_q6:.4f}  ({risky_clean_q6 / 10:.4f}%)")
print(f"\nMarket clean price   : EUR {MKT_CLEAN_EUR:.4f}  ({MKT_CLEAN_PCT:.2f}%)")
print(f"Market gross price   : EUR {MKT_GROSS_EUR:.4f}")

# =============================================================================
# Price comparison table
# =============================================================================
print("\n" + "=" * 66)
print("Q10 -- Market Price Comparison")
print("=" * 66)
print(f"\n{'Price measure':<40} {'EUR':>9}  {'% par':>7}")
print("-" * 60)
print(f"{'RF fair value (DB model, Q5)':<40} {rf_npv_clean:>9.2f}  {rf_npv_clean / 10:>7.2f}%")
print(f"{'Risky fair value (CVA-adj., Q6)':<40} {risky_clean_q6:>9.2f}  {risky_clean_q6 / 10:>7.2f}%")
print(f"{'EuroTLX market price':<40} {MKT_CLEAN_EUR:>9.2f}  {MKT_CLEAN_PCT:>7.2f}%")
print("-" * 60)
print(f"{'Market vs RF fair value':<40} {MKT_CLEAN_EUR - rf_npv_clean:>+9.2f}  "
      f"{(MKT_CLEAN_PCT - rf_npv_clean / 10):>+7.2f}pp")
print(f"{'Market vs risky fair value':<40} {MKT_CLEAN_EUR - risky_clean_q6:>+9.2f}  "
      f"{(MKT_CLEAN_PCT - risky_clean_q6 / 10):>+7.2f}pp")

# =============================================================================
# Market-implied CDS spread -- exact solve against the SAME model used
# throughout, rather than the risk-free reference silently being a
# different (non-option-adjusted) valuation than Q6's.
# =============================================================================
v_at_zero = compute_risky_gross(0.0, RECOVERY_MKT)
print(f"\nV_risky(s=0 bp)   = EUR {v_at_zero:.4f}  (market gross = EUR {MKT_GROSS_EUR:.4f})")

MOD_DUR = 6.8   # approximate, consistent with the LaTeX writeup -- kept
                # only as a secondary cross-check on the exact solve below.

if v_at_zero < MKT_GROSS_EUR:
    print("\nMarket price EXCEEDS the risk-free model price even at zero credit spread.")
    print("No positive implied CDS spread exists under this model.")
    delta_P    = (MKT_CLEAN_PCT - rf_npv_clean / 10) / 100.0
    delta_y_bp = -(delta_P / MOD_DUR) * 1e4
    s_star     = None
    print(f"  Duration-approx implied delta_y : {delta_y_bp:+.1f} bp (effectively zero / negative)")
else:
    def objective(s):
        return compute_risky_gross(s, RECOVERY_MKT) - MKT_GROSS_EUR

    s_star   = brentq(objective, 0.0, 0.50, xtol=1e-10)
    lam_star = s_star / (1.0 - RECOVERY_MKT)
    print(f"\nMarket-implied CDS spread (exact, full reval) : {s_star * 1e4:.2f} bp")
    print(f"Market-implied lambda                         : {lam_star:.6f} p.a.")

    # Cross-check against the old linear-duration approximation
    delta_P    = (MKT_CLEAN_PCT - rf_npv_clean / 10) / 100.0
    delta_y_bp = -(delta_P / MOD_DUR) * 1e4
    print(f"Duration-approx cross-check                   : {delta_y_bp:+.1f} bp  "
          f"(diff vs exact: {delta_y_bp - s_star * 1e4:+.1f} bp)")

# =============================================================================
# Implied spread sensitivity to recovery rate
# =============================================================================
print("\n" + "=" * 66)
print("Implied credit spread sensitivity to recovery rate")
print(f"  Market gross price = EUR {MKT_GROSS_EUR:.4f}")
print("=" * 66)
print(f"  {'Recovery':>10} {'Lambda':>12} {'CVA EUR':>10} {'Risky Clean EUR':>16} {'Implied spread':>15}")
print("-" * 66)

for rec in [0.20, 0.30, 0.40, 0.50, 0.60]:
    v0 = compute_risky_gross(0.0, rec)
    risky_g_r = compute_risky_gross(CDS_MARKET_BP / 1e4, rec)
    risky_c_r = risky_g_r - AI
    if v0 >= MKT_GROSS_EUR:
        s_impl = brentq(lambda s, r=rec: compute_risky_gross(s, r) - MKT_GROSS_EUR, 0.0, 0.50, xtol=1e-10)
        lam_r  = s_impl / (1.0 - rec)
        s_label = f"{s_impl * 1e4:.2f} bp"
    else:
        lam_r   = (CDS_MARKET_BP / 1e4) / (1.0 - rec)
        s_label = "no root (mkt > RF)"
    marker = "  <-- base" if rec == RECOVERY_MKT else ""
    print(f"  {rec*100:>9.0f}%  {lam_r:>12.6f}  {(rf_npv - risky_g_r):>10.4f}  "
          f"{risky_c_r:>16.4f}  {s_label:>15}{marker}")

# =============================================================================
# Bond-CDS basis (exact solve vs quoted CDS market spread)
# =============================================================================
print("\n" + "=" * 66)
print("Bond-CDS Basis")
print("=" * 66)
if s_star is not None:
    basis_bp = s_star * 1e4 - CDS_MARKET_BP
    print(f"  CDS market spread              : {CDS_MARKET_BP:+.2f} bp")
    print(f"  Bond-implied spread (exact)     : {s_star * 1e4:+.2f} bp")
    print(f"  Bond-CDS basis                  : {basis_bp:+.1f} bp  "
          f"({'bond rich vs CDS' if basis_bp < 0 else 'bond cheap vs CDS'})")
else:
    basis_bp = delta_y_bp - CDS_MARKET_BP
    print(f"  CDS market spread               : {CDS_MARKET_BP:+.2f} bp")
    print(f"  Bond-implied spread (duration approx): {delta_y_bp:+.1f} bp")
    print(f"  Bond-CDS basis (approx)         : {basis_bp:+.1f} bp  "
          f"({'bond rich vs CDS' if basis_bp < 0 else 'bond cheap vs CDS'})")

# =============================================================================
# Summary table
# =============================================================================
print("\n" + "=" * 66)
print("FINAL SUMMARY  (EUR per 1000 nominal, trade date 05-Nov-2025)")
print("=" * 66)
print(f"{'Metric':<40} {'Value':>15}")
print("-" * 58)
print(f"{'RF clean price (Q5)':<40} {rf_npv_clean / 10:>14.2f}%")
print(f"{'Risky clean price (CDS=' + f'{CDS_MARKET_BP:.0f}bp)':<40} {risky_clean_q6 / 10:>14.2f}%")
print(f"{'EuroTLX market clean price':<40} {MKT_CLEAN_PCT:>14.2f}%")
print(f"{'Market CDS spread':<40} {f'{CDS_MARKET_BP:.0f} bp':>15}")
implied_label = f"{s_star*1e4:+.1f} bp" if s_star is not None else f"{delta_y_bp:+.1f} bp (approx)"
print(f"{'Bond-implied spread':<40} {implied_label:>15}")
print(f"{'Bond-CDS basis':<40} {basis_bp:>+14.1f} bp")
print(f"{'Market vs RF (price gap)':<40} {MKT_CLEAN_EUR - rf_npv_clean:>+14.4f} EUR")
print(f"{'Market vs risky (price gap)':<40} {MKT_CLEAN_EUR - risky_clean_q6:>+14.4f} EUR")

# =============================================================================
# Save outputs
# =============================================================================
summary = {
    "Metric": [
        "RF clean price (EUR)", "RF clean price (%)",
        "Risky clean price (EUR)", "Risky clean price (%)",
        "Market clean price (EUR)", "Market clean price (%)",
        "Accrued interest (EUR)", "CDS spread (bp)",
        "Bond-implied spread (bp)", "Bond-CDS basis (bp)",
        "Market vs RF gap (EUR)", "Market vs risky gap (EUR)",
    ],
    "Value": [
        round(rf_npv_clean, 4), round(rf_npv_clean / 10, 4),
        round(risky_clean_q6, 4), round(risky_clean_q6 / 10, 4),
        round(MKT_CLEAN_EUR, 4), round(MKT_CLEAN_PCT, 4),
        round(AI, 4), round(CDS_MARKET_BP, 2),
        round(s_star * 1e4, 2) if s_star is not None else round(delta_y_bp, 2),
        round(basis_bp, 2),
        round(MKT_CLEAN_EUR - rf_npv_clean, 4), round(MKT_CLEAN_EUR - risky_clean_q6, 4),
    ],
}
pd.DataFrame(summary).to_csv(os.path.join(_here, os.pardir, "data", "q10_summary.csv"), index=False)
print("\nSaved: q10_summary.csv")



Risk-free NPV  gross : EUR 1102.2711  (110.2271%)
Risk-free NPV  clean : EUR 1097.4917  (109.7492%)
Accrued interest     : EUR 4.7794

Risky NPV at CDS=72bp   gross : EUR 1049.1088  (104.9109%)
Risky NPV at CDS=72bp   clean : EUR 1044.3294  (104.4329%)

Market clean price   : EUR 1008.1000  (100.81%)
Market gross price   : EUR 1012.8794

Q10 -- Market Price Comparison

Price measure                                  EUR    % par
------------------------------------------------------------
RF fair value (DB model, Q5)               1097.49   109.75%
Risky fair value (CVA-adj., Q6)            1044.33   104.43%
EuroTLX market price                       1008.10   100.81%
------------------------------------------------------------
Market vs RF fair value                     -89.39    -8.94pp
Market vs risky fair value                  -36.23    -3.62pp

V_risky(s=0 bp)   = EUR 1102.2711  (market gross = EUR 1012.8794)

Market-implied CDS spread (exact, full reval) : 125.51 bp
Market-impli

         20%      0.015340     53.8166         1043.6751        122.72 bp


         30%      0.017699     53.5348         1043.9569        123.89 bp
         40%      0.020918     53.1623         1044.3294        125.51 bp  <-- base


         50%      0.025574     52.6467         1044.8450        127.87 bp


         60%      0.032910     51.8860         1045.6057        131.64 bp

Bond-CDS Basis
  CDS market spread              : +72.00 bp
  Bond-implied spread (exact)     : +125.51 bp
  Bond-CDS basis                  : +53.5 bp  (bond cheap vs CDS)

FINAL SUMMARY  (EUR per 1000 nominal, trade date 05-Nov-2025)
Metric                                             Value
----------------------------------------------------------
RF clean price (Q5)                              109.75%
Risky clean price (CDS=72bp)                     104.43%
EuroTLX market clean price                       100.81%
Market CDS spread                                  72 bp
Bond-implied spread                            +125.5 bp
Bond-CDS basis                                    +53.5 bp
Market vs RF (price gap)                       -89.3917 EUR
Market vs risky (price gap)                    -36.2294 EUR

Saved: q10_summary.csv
